# Lab 13 - Inference Service Optimization

**Textbook:** Chip Huyen, *AI Engineering*, Chapter 9 "Inference Optimization", second half: "Inference Optimization" (Model Optimization: model compression; Inference Service Optimization: batching, prompt caching).
**Environment:** cluster notebook (your local Docker JupyterLab on the KU VPN).
**Time:** Part A about 2.5 hours. This is a **Part-A-only week** (side project 3 is due). Part B is an optional stretch for bonus points.

In Lab 12 you measured how fast the cluster is. This week you make Larkspur's help-desk assistant **faster and cheaper without buying hardware**, using the service-level levers from Chapter 9: prompt caching, batching, output length control, and (on paper) quantization. Every claim you make in the analysis should rest on a number you measured here.

### Before you start
- Connect to the **KU VPN** and start your course container.
- Read the second half of Chapter 9 (the section "Inference Optimization": "Model Optimization" and "Inference Service Optimization"). Keep it open for section names.
- The cluster is shared and some cells send long (about 6,000-token) prompts. Run each experiment cell **once**; Part A sends about 50 requests.
- Cells use top-level `await`; run them in order.

## Words you'll meet in this lab

You don't need to memorize these. Each one is explained again where it first matters; come back here if a word stops you.

| Term | What it means | Why it matters |
|---|---|---|
| **TTFT, TPOT, prefill, decode** | From Lab 12. TTFT: seconds until the first token arrives. TPOT: seconds per token after that. Prefill: the model reads the whole prompt. Decode: it writes the answer one token at a time. | Every optimization this week speeds up one of these. |
| **KV cache** | The keys and values the model computes for every token it has read (from Lab 02). | Prompt caching works by saving and reusing it. |
| **Prefix** | The beginning part of a prompt, from the first token up to some point. | A cache can only reuse a prompt's prefix, so what comes first matters. |
| **Prompt caching** | The server saves the work it did on a prompt's prefix and reuses it when a new prompt starts the same way, skipping that part of prefill. | It can cut TTFT sharply for long, repeated instructions. |
| **Cold / warm** | A cold request has nothing in the cache to reuse; a warm one starts with a prefix the server has seen recently. | Comparing them shows how much caching saves. |
| **Backend** | The machine that actually answered a request (here, which Mac Studio). | Each Mac has its own cache, so the backend decides whether a request is warm. |
| **Continuous batching** | The server works on several requests together in one pass over the weights; new requests join and finished ones leave at any time. | It is why sending requests concurrently finishes a pile of work sooner. |
| **Sequential vs concurrent** | Sequential: one request after another. Concurrent: several in progress at the same time. | Same work, very different wall-clock time. |
| **`max_tokens`, `finish_reason`** | `max_tokens` is a hard limit on answer length. `finish_reason` says why an answer stopped: `stop` (the model finished) or `length` (it hit the limit and was cut off). | Shorter answers are faster and cheaper, but cut-off answers are bad answers. |
| **Input vs output tokens (pricing)** | Hosted APIs charge separately for tokens you send and tokens the model writes; output costs more (from Lab 01). | Output length drives both cost and latency. |
| **Quantization** | Storing weights with fewer bits (from Lab 02). | Fewer bytes per weight means faster decode and less memory, with some quality risk. |
| **Decode ceiling** | The fastest decode the memory bandwidth allows: bandwidth divided by the bytes of active weights read per token (from Labs 02 and 12). | Lets you compare quantization choices without running them. |
| **Response cache (exact / semantic)** | Stores past answers. Exact: reuse an answer only for the same question. Semantic: also reuse it for a question with similar meaning. | A hit skips the model completely. |
| **Embedding, cosine similarity** | An embedding is a list of numbers that captures a text's meaning (from Lab 08). Cosine similarity scores how alike two embeddings are: 1.0 is the same direction, near 0 unrelated. | The semantic cache decides "same question?" with this score. |
| **Threshold, false hit** | The threshold is the similarity score a match must reach. A false hit is a match that returns the answer to a *different* question. | Lower thresholds give more hits, and more wrong ones. |

## Setup

In [ ]:
import asyncio, time, uuid, re, json, datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from openai import AsyncOpenAI
from itec632 import get_client, MODELS, NO_THINK, settings
from itec632.corpus import load_docs, corpus_dir
from itec632.grading import LabChecker

ck = LabChecker("lab13", part="cluster")
ckb = LabChecker("lab13", part="bonus")    # Part B stretch checks (reported as bonus in the summary)
client = get_client()
base_url, api_key = settings()
aclient = AsyncOpenAI(base_url=base_url, api_key=api_key, timeout=300, max_retries=0)

MOE, DENSE, SMALL, EMBED = MODELS["main"], MODELS["dense"], MODELS["small"], MODELS["embed"]
COLORS = {MOE: "#2a78d6", DENSE: "#eb6834", SMALL: "#1baf7a"}
PAUSE_S = 1.5         # between short requests
LONG_PAUSE_S = 3.0    # between ~6K-token requests (keeps you under 300k tokens/minute)

available = [m.id for m in client.models.list().data]
ck.check("connected", all(m in available for m in [MOE, SMALL, EMBED]), 1,
         "Could not see qwen-moe, qwen-small and embed. Check the VPN and your .env.")
print("Started (UTC):", dt.datetime.now(dt.timezone.utc).strftime("%A %Y-%m-%d %H:%M"))

The timing helper is the same idea as Lab 12 (streaming, `time.perf_counter()`, token counts from `stream_options={"include_usage": True}` with an estimate as fallback). It is complete here. It also records `backend` (which Mac served the request, from the gateway's `x-litellm-model-id` header, a label the server attaches to its reply) and `cached_tokens` (how many prompt tokens the server reused from its cache) if the server reports them.

In [ ]:
def compute_metrics(t_start, t_first, t_end, n_out):
    """Turn the timestamps of one streamed request into latency metrics.

    t_start: perf_counter() just before the request was sent
    t_first: perf_counter() when the first token arrived
    t_end:   perf_counter() when the stream finished
    n_out:   number of output tokens (usage.completion_tokens)
    """
    ttft = t_first - t_start
    total = t_end - t_start
    tpot = (t_end - t_first) / (n_out - 1) if n_out and n_out > 1 else None
    out_tok_per_s = n_out / total if total > 0 else None
    return {"ttft_s": ttft, "tpot_s": tpot, "total_s": total, "out_tok_per_s": out_tok_per_s}

In [ ]:
BUDGET = {"sent": 0}   # counts every request this notebook sends (the cluster is shared!)


def _estimate_tokens(text):
    """Rough fallback: about 4 characters per token for English text."""
    return max(1, round(len(text or "") / 4))


async def measure_stream(model, messages, max_tokens=200, include_usage=True, extra_body=None):
    """Send ONE streamed chat request and time it. Returns a dict (one row of results).

    Timing uses time.perf_counter(), a monotonic high-resolution clock made for
    measuring intervals (time.time() can jump if the system clock is adjusted).
    """
    rec = {"model": model, "max_tokens": max_tokens, "ok": False, "error": None,
           "ttft_s": None, "tpot_s": None, "total_s": None, "out_tok_per_s": None,
           "prompt_tokens": None, "completion_tokens": None, "cached_tokens": None,
           "usage_estimated": False, "est_completion_tokens": None, "n_chunks": 0,
           "finish_reason": None, "backend": None, "text": "", "chunk_times": [],
           "sent_utc": dt.datetime.now(dt.timezone.utc).strftime("%H:%M:%S")}
    kwargs = dict(model=model, messages=messages, max_tokens=max_tokens, stream=True,
                  extra_body=NO_THINK if extra_body is None else extra_body)
    if include_usage:
        # Ask the server to append one final chunk with exact token counts.
        kwargs["stream_options"] = {"include_usage": True}
    BUDGET["sent"] += 1
    usage, parts, t_first = None, [], None
    t_start = time.perf_counter()
    try:
        # with_raw_response gives us the HTTP headers too (which backend Mac served us).
        raw = await aclient.chat.completions.with_raw_response.create(**kwargs)
        rec["backend"] = raw.headers.get("x-litellm-model-id") or raw.headers.get("x-litellm-model-api-base")
        async for chunk in raw.parse():
            now = time.perf_counter()
            if getattr(chunk, "usage", None):
                usage = chunk.usage                      # the final usage chunk
            for choice in chunk.choices or []:          # the usage chunk has no choices
                delta = choice.delta
                piece = getattr(delta, "content", None) or ""
                thought = getattr(delta, "reasoning_content", None) or ""
                if piece or thought:
                    if t_first is None:
                        t_first = now                    # first token of any kind
                    rec["chunk_times"].append(round(now - t_start, 4))
                    parts.append(piece)
                if choice.finish_reason:
                    rec["finish_reason"] = choice.finish_reason
        t_end = time.perf_counter()
    except Exception as e:                               # e.g. RateLimitError (429), timeout
        rec["total_s"] = time.perf_counter() - t_start
        rec["error"] = f"{type(e).__name__}: {e}"[:200]
        return rec
    text = "".join(parts)
    rec["text"], rec["n_chunks"] = text, len(rec["chunk_times"])
    rec["est_completion_tokens"] = _estimate_tokens(text)
    if usage is not None and usage.completion_tokens:
        rec["prompt_tokens"] = usage.prompt_tokens
        rec["completion_tokens"] = usage.completion_tokens
        details = getattr(usage, "prompt_tokens_details", None)
        rec["cached_tokens"] = getattr(details, "cached_tokens", None) if details else None
    else:
        # No usage chunk (stream cut off, or a server that ignores stream_options):
        # fall back to an estimate and flag it so you know the number is approximate.
        rec["usage_estimated"] = True
        rec["prompt_tokens"] = sum(_estimate_tokens(m["content"]) for m in messages)
        rec["completion_tokens"] = rec["est_completion_tokens"]
    if t_first is None:
        rec["error"] = "stream ended without any content"
        rec["total_s"] = t_end - t_start
        return rec
    rec.update(compute_metrics(t_start, t_first, t_end, rec["completion_tokens"]))
    rec["ok"] = True
    return rec


def p50_p90(df, by, cols):
    """Median (p50) and 90th percentile (p90) of each column, per group. Failed requests are skipped."""
    rows = []
    ok = df[df["ok"] == True] if len(df) else df  # noqa: E712
    for key, g in ok.groupby(by):
        row = dict(zip(by, key)) if isinstance(by, list) else {by: key}
        row["n"] = len(g)
        for c in cols:
            s = pd.to_numeric(g[c], errors="coerce").dropna()
            row[f"{c}_p50"] = s.median() if len(s) else np.nan
            row[f"{c}_p90"] = s.quantile(0.9) if len(s) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def label_backends(df):
    """Replace long backend ids from the gateway headers with short labels: mac-A, mac-B, ..."""
    names = {}
    for b in df["backend"]:
        if b and b not in names:
            names[b] = f"mac-{chr(65 + len(names))}"
    return df["backend"].map(lambda b: names.get(b, "unknown"))


def nonce():
    """A short random tag. Putting it at the START of a prompt makes the prompt unique,
    so the server cannot reuse a cached prefix from an earlier request."""
    return uuid.uuid4().hex[:8]

## Part A - Guided practice

### A1. Prompt caching: reuse the prefix, skip the prefill

Larkspur's assistant sends the same long instructions and reference documents with every question; only the short question at the end changes. Chapter 9 ("Prompt caching" under "Inference Service Optimization") describes the fix: store the processed prompt (its **KV cache**) and reuse it when a new prompt **starts with** the same tokens (the same **prefix**). Prefill was the expensive, compute-bound part of TTFT in Lab 12, so skipping it should cut TTFT sharply.

oMLX (the program that runs the models on each Mac) keeps such a prefix cache. Two things decide whether you benefit:

1. **Order.** The server can only reuse the cache up to the **first token that differs**. Why? Each token's stored keys and values are computed while that token "looks at" all the tokens before it (attention, from Lab 02). So the values stored for token 50 already carry information about tokens 1-49. If token 5 changes, everything from token 5 on was computed in a different context and can't be reused, even if the later text is word-for-word the same. Shared material must therefore come first, and the part that changes last:

   ```
   Prompt 1: [Session W][instructions][documents ..........][question 1]
   Prompt 2: [Session W][instructions][documents ..........][question 2]
             <------------ identical: reused -------------><-- new -->

   Prompt 3: [Session W][instructions][question 3][documents ..........]
   Prompt 4: [Session W][instructions][question 4][documents ..........]
             <------- reused ------->^ first difference: everything after it is recomputed
   ```
2. **Which machine.** Each Mac has its own cache. `qwen-moe` runs on two Macs and the gateway picks one at random per request, so a "warm" prompt may land on the Mac that has never seen it. `qwen-small` has only one Mac.

The reference pack below is three current Larkspur documents (price sheet, SLA, service catalog), about 6,000 tokens. Each prompt starts with `Session <tag>`. A fresh tag makes the prompt new, so it is **cold** (no classmate can have cached it either); reusing a tag makes it **warm**.

**TODO:** write `prefix_last()`: same content, but the **question comes before** the reference documents.

In [ ]:
by_id = {d["doc_id"]: d for d in load_docs()}
PACK = "\n\n".join(f"## {by_id[i]['title']} ({i})\n{by_id[i]['text']}" for i in ["PRICE-2026", "SLA-EXH-2026", "SVC-CAT-2026"])[:24000]
INSTRUCTIONS = ("You are Larkspur Ridge IT's help-desk assistant. Answer the question using only the "
                "reference documents. Be brief.")
print(f"Reference pack: {len(PACK):,} characters (about {len(PACK) // 4:,} tokens)")


def prefix_first(tag, question):
    """Shared material FIRST (cache-friendly): instructions + documents, then the question."""
    return [{"role": "system", "content": f"Session {tag}. {INSTRUCTIONS}\n\n=== REFERENCE DOCUMENTS ===\n{PACK}"},
            {"role": "user", "content": question}]

In [ ]:
def prefix_last(tag, question):
    """Question FIRST, then the documents (the cache-unfriendly order)."""
    # TODO: return two messages (like prefix_first): a system message with the same session line and
    #       INSTRUCTIONS but no documents, and a user message with the question FIRST and then PACK.
    return None

In [ ]:
_t = prefix_last("abc", "QQQ?")
ok_last = bool(_t) and "QQQ?" in _t[-1]["content"] and PACK[:200] in _t[-1]["content"] \
          and _t[-1]["content"].index("QQQ?") < _t[-1]["content"].index(PACK[:200]) and PACK[:200] not in _t[0]["content"]
ck.check("prefix_last_built", ok_last, 2,
         "prefix_last() must return [system, user] with the question before PACK in the user message, and no PACK in system.");

Now the experiment (about 21 long requests, 2-3 minutes). We only ask for 40 output tokens because we care about TTFT.

- **cold**: a new tag every time, documents first.
- **warm**: first one **prime** request with tag W ("Reply with OK."). Its only job is to put the documents into the cache; its answer is thrown away. Then 6 different questions with the same tag W, which can reuse that cache.
- **last**: question before documents, one prime, then 4 questions with the same tag.
- **qwen-small** warm vs cold, for a model with a single backend.

**TODO (predict):** before you run the experiment, guess how many times faster a warm `qwen-moe` request will be than a cold one (median cold TTFT divided by median warm TTFT). Put your guess in `MY_SPEEDUP_GUESS` (for example `3` for three times faster). Think about how much of the prompt the warm requests can reuse. You'll compare it with the measured speedup.

In [ ]:
# TODO: your guess, a number such as 3 (median cold TTFT / median warm TTFT for qwen-moe)
MY_SPEEDUP_GUESS = None

In [ ]:
CACHE_QS = [
    "What is the support phone number?", "How much is the Premier tier per user?", "What are the payment terms?",
    "What is the trip charge in the metro?", "What is the PagerLine number?", "Which holidays does Larkspur observe?",
    "What is the P2 response target on Essentials?", "How are SLA credits requested?",
    "What is the business-hours labor rate?", "When are invoices sent?",
]
cache_records = []


async def run_cache(model, condition, messages):
    rec = await measure_stream(model, messages, max_tokens=40)
    rec["condition"] = condition
    cache_records.append(rec)
    await asyncio.sleep(LONG_PAUSE_S)
    return rec

for q in CACHE_QS[:4]:                                         # cold
    await run_cache(MOE, "cold", prefix_first(nonce(), q))
WARM_TAG = nonce()                                             # warm (this tag is reused in A3 and Part B)
await run_cache(MOE, "prime", prefix_first(WARM_TAG, "Reply with OK."))
for q in CACHE_QS[4:10]:
    await run_cache(MOE, "warm", prefix_first(WARM_TAG, q))
if prefix_last("x", "y") is not None:                         # question first
    LAST_TAG = nonce()
    await run_cache(MOE, "prime_last", prefix_last(LAST_TAG, "Reply with OK."))
    for q in CACHE_QS[:4]:
        await run_cache(MOE, "last", prefix_last(LAST_TAG, q))
else:
    print("Skipping the 'last' condition: complete prefix_last() first.")
await run_cache(SMALL, "cold", prefix_first(nonce(), CACHE_QS[0]))   # single-backend model
await run_cache(SMALL, "prime", prefix_first(WARM_TAG, "Reply with OK."))
for q in CACHE_QS[1:4]:
    await run_cache(SMALL, "warm", prefix_first(WARM_TAG, q))

df_cache = pd.DataFrame(cache_records)
df_cache["backend"] = label_backends(df_cache)
print("errors:", int((~df_cache["ok"]).sum()), "| requests so far:", BUDGET["sent"])
display(df_cache[["model", "condition", "backend", "prompt_tokens", "cached_tokens", "ttft_s", "total_s"]].round(3))

**Reading the table:** one row per request, in the order sent. `condition` is the experiment group (`prime` rows are the requests that fill the cache). `backend` is which Mac answered (`mac-A`, `mac-B`, or `unknown` if the gateway didn't say). `prompt_tokens` is the prompt size; `cached_tokens` is how many of those the server reused from its cache (empty if the server doesn't report it). `ttft_s` and `total_s` are the timings in seconds.

In [ ]:
cache_summary = p50_p90(df_cache, ["model", "condition"], ["ttft_s"])
display(cache_summary.round(3))

fig, ax = plt.subplots(figsize=(8, 3.6))
order = ["cold", "prime", "warm", "prime_last", "last"]
markers = {"mac-A": "o", "mac-B": "s", "unknown": "o"}
for m in [MOE, SMALL]:
    for i, cond in enumerate(order):
        g = df_cache[(df_cache.model == m) & (df_cache.condition == cond) & df_cache.ok]
        for _, row in g.iterrows():
            ax.scatter(i + (0.12 if m == SMALL else -0.12), row.ttft_s, color=COLORS[m],
                       marker=markers.get(row.backend, "o"), s=45, alpha=0.85)
for m in [MOE, SMALL]:
    ax.scatter([], [], color=COLORS[m], label=m)
for b, mk in markers.items():
    if b in set(df_cache.backend) and b != "unknown":
        ax.scatter([], [], color="#777777", marker=mk, label=b)
ax.set_xticks(range(len(order)), order); ax.set_ylabel("TTFT (s)"); ax.set_ylim(bottom=0)
ax.set_title("TTFT by cache condition (marker shape = which Mac served it)"); ax.grid(alpha=0.3, axis="y"); ax.legend(fontsize=8)
plt.show()

**Reading the table and plot:** the table gives, for each model and condition, the number of successful requests (`n`) and the median (`ttft_s_p50`) and p90 (`ttft_s_p90`) TTFT. In the plot, each dot is one request: condition across, TTFT up. Color is the model and marker shape is the Mac that answered, so a slow dot among the warm ones with a different shape probably landed on the Mac without the cache.

**Check your prediction.** The next cell computes the speedup for you: the median cold TTFT divided by the median warm TTFT for `qwen-moe` (a speedup of 4 means warm requests got their first token 4 times sooner). It prints your guess next to it. Then look at the warm rows' `backend` column in the table above: did any warm request land on a Mac that had not seen the prefix yet? What was its TTFT? You'll use both in Q1.

In [ ]:
def med(model, cond, col="ttft_s"):
    g = df_cache[(df_cache.model == model) & (df_cache.condition == cond) & df_cache.ok]
    return g[col].median() if len(g) else None

cold_moe, warm_moe = med(MOE, "cold"), med(MOE, "warm")
cache_speedup = cold_moe / warm_moe if cold_moe and warm_moe else None     # None if a condition has no successful requests
print("qwen-moe cache speedup (cold / warm median TTFT):", None if cache_speedup is None else round(cache_speedup, 2),
      "  your guess:", MY_SPEEDUP_GUESS)

In [ ]:
n_cond = df_cache[df_cache.ok].groupby(["model", "condition"]).size()
ck.check("cache_conditions_measured", n_cond.get((MOE, "cold"), 0) >= 3 and n_cond.get((MOE, "warm"), 0) >= 5
         and n_cond.get((MOE, "last"), 0) >= 3, 2,
         "Need >= 3 cold, >= 5 warm and >= 3 'last' qwen-moe requests that succeeded. Complete prefix_last() and re-run.")
ck.check("cache_prediction_made", isinstance(MY_SPEEDUP_GUESS, (int, float)) and MY_SPEEDUP_GUESS > 0, 1,
         "Put your guess in MY_SPEEDUP_GUESS (a positive number such as 3) before running the cache experiment.")
if isinstance(MY_SPEEDUP_GUESS, (int, float)):
    ck.record("cache_speedup_guess", float(MY_SPEEDUP_GUESS))
# Not graded (the gateway's random choice of Mac decides part of it): were warm requests clearly faster?
warm_ok = bool(cold_moe and warm_moe and warm_moe < 0.8 * cold_moe)
ck.record("warm_faster_than_cold", warm_ok)
if cold_moe and warm_moe and not warm_ok:
    print("Note: warm requests were not clearly faster this time. Not graded. Look at the backend column: several warm")
    print("requests may have landed on the Mac that never saw the prefix (this can happen by chance). Discuss it in Q1.")
for (m, cond), v in cache_summary.set_index(["model", "condition"])["ttft_s_p50"].items():
    ck.record(f"ttft_{cond}_{m.replace('qwen-', '')}", float(v))
if cache_speedup:
    ck.record("cache_speedup_moe", float(cache_speedup))

### A2. Batching: sequential vs concurrent

Chapter 9 ("Batching") compares static batching (wait until a batch is full), dynamic batching (wait until it is full or a time limit passes) and **continuous batching**. oMLX uses continuous batching: new requests join the running batch as soon as there is a free slot (up to 8 per Mac), and finished ones leave without waiting for the rest. Why is that cheap? Decode is memory-bandwidth-bound (Lab 12): the slow part of each step is reading the weights from memory, not the math. Think of a bus that drives the same route whether it carries 1 passenger or 4. The trip (one read of the weights) costs about the same, so taking 4 passengers per trip moves everyone much faster. Each passenger's ride gets a little slower, but the whole group arrives far sooner.

For an application this means: if Larkspur needs to process many independent items (summarize 8 tickets, classify a queue), **send them concurrently** instead of in a loop. Here you process the same 8 short requests both ways. Your key allows 4 at a time, so the concurrent version sends two groups of 4.

`asyncio.gather(...)` (you used it in Lab 12's `run_burst`) means "start all of these requests now, then wait until every one is done". The `*` in front of the list unpacks it, so each request becomes its own argument.

**TODO 1:** before you run the cell, predict: the concurrent version's wall time will be about what fraction of the sequential version's? Put your guess in `MY_PREDICTION` (for example `0.5` for half). You'll compare it with the real ratio.

**TODO 2:** in the concurrent loop, send each group of 4 at once with `asyncio.gather`.

In [ ]:
TICKETS = [
    "Outlook keeps asking for a password after a password change.", "A shared drive mapping disappeared after a reboot.",
    "The office printer shows offline for everyone.", "A new hire needs a laptop and accounts by Monday.",
    "Teams calls drop every few minutes on Wi-Fi.", "A user deleted a folder on the file server.",
    "The firewall sends a high CPU alert every night.", "A user's OneDrive will not sync.",
]


def ticket_msgs(t):
    return [{"role": "system", "content": f"Request {nonce()}. You are Larkspur Ridge IT's help-desk assistant."},
            {"role": "user", "content": f"Ticket: {t}\nWrite three short troubleshooting steps for a Tier 1 technician."}]

# TODO 1: your prediction, a number such as 0.5 (concurrent wall time as a fraction of sequential wall time)
MY_PREDICTION = None

t0 = time.perf_counter()
seq_recs = []
for t in TICKETS:
    seq_recs.append(await measure_stream(MOE, ticket_msgs(t), max_tokens=150))
seq_wall = time.perf_counter() - t0
await asyncio.sleep(5)

t0 = time.perf_counter()
conc_recs = []
for group in (TICKETS[:4], TICKETS[4:]):
    # TODO 2: results = await asyncio.gather(*[measure_stream(MOE, ticket_msgs(t), max_tokens=150) for t in group])
    results = []
    conc_recs += list(results)
conc_wall = time.perf_counter() - t0

In [ ]:
df_batch = pd.concat([pd.DataFrame(seq_recs).assign(mode="sequential"), pd.DataFrame(conc_recs).assign(mode="concurrent")],
                     ignore_index=True)
walls = {"sequential": seq_wall, "concurrent": conc_wall}
batch_table = p50_p90(df_batch, "mode", ["ttft_s", "total_s", "tpot_s"])
if len(batch_table):
    tok = df_batch[df_batch.ok].groupby("mode")["completion_tokens"].sum()
    batch_table["wall_s"] = batch_table["mode"].map(walls)
    batch_table["throughput_tok_s"] = batch_table["mode"].map(lambda m: tok.get(m, 0) / walls[m])
    display(batch_table.round(3))
if seq_wall and conc_wall:
    print(f"Actual ratio (concurrent / sequential wall time): {conc_wall / seq_wall:.2f}   your prediction: {MY_PREDICTION}")
ck.check("batch_prediction_made", isinstance(MY_PREDICTION, (int, float)) and 0 < MY_PREDICTION <= 1.5, 1,
         "Put your guess in MY_PREDICTION before running the batching cell (a number such as 0.5).")
if isinstance(MY_PREDICTION, (int, float)):
    ck.record("batch_prediction", float(MY_PREDICTION))
ok_counts = df_batch[df_batch.ok].groupby("mode").size() if len(df_batch) else {}
ck.check("batching_ran", ok_counts.get("sequential", 0) == 8 and ok_counts.get("concurrent", 0) == 8, 2,
         "Both modes need 8 successful requests. Complete the asyncio.gather TODO.")
ck.check("concurrent_faster", ok_counts.get("concurrent", 0) == 8 and conc_wall < 0.8 * seq_wall, 2,
         "Concurrent wall time should be clearly below sequential. Are the 4 requests of a group really sent together?")
ck.record("batch_wall_sequential_s", float(seq_wall)); ck.record("batch_wall_concurrent_s", float(conc_wall));

**Reading the table:** one row per mode. The `ttft_s`, `total_s` and `tpot_s` columns (median `_p50` and `_p90`) are what each single request experienced. `wall_s` is the clock time to finish all 8 tickets, and `throughput_tok_s` is all output tokens divided by that wall time. Compare the two kinds of numbers: per-request latency can get a little worse while the whole job finishes much sooner.

### A3. Output length control: `max_tokens` vs asking for concise answers

Lab 12 showed `total latency = TTFT + TPOT x (output tokens - 1)`. With a warm prompt cache, TTFT is small, so **output length dominates latency**, and output tokens are also the expensive tokens on every price sheet. There are two levers:

- **`max_tokens`** is a hard stop. It bounds latency and cost, but the model does not know about it, so answers are cut off mid-sentence (`finish_reason == "length"`).
- **A concise-answer instruction** changes what the model plans to say. Answers end naturally, but length is not guaranteed.

We use the warm reference pack from A1 (same tag, so TTFT stays low) and these hypothetical prices, in USD per million tokens, so you can talk about cost. (They are made up for this lab, in line with hosted APIs; other labs may use simpler prices.)

`request_cost()` is provided: it multiplies the input tokens by the input price and the output tokens by the output price, adds them, and divides by one million (the prices are per million tokens).

**TODO:** write your concise-answer instruction in `CONCISE_STYLE`. After the results, you'll decide which setting you would ship.

In [ ]:
# Hypothetical prices (USD per 1M tokens: input, output), in line with hosted APIs for models of these sizes.
PRICE_PER_M = {MOE: (0.20, 0.80), DENSE: (0.40, 1.60), SMALL: (0.05, 0.20)}


def request_cost(model, prompt_tokens, completion_tokens):
    """Cost in USD of one request at PRICE_PER_M."""
    p_in, p_out = PRICE_PER_M[model]
    return (prompt_tokens * p_in + completion_tokens * p_out) / 1_000_000

# TODO: an instruction that makes answers short (a limit on the number of sentences works better than "be brief")
CONCISE_STYLE = ""

In [ ]:
VERBOSE_STYLE = "Answer thoroughly, with background and a full explanation."
CONDITIONS = {
    "verbose":         {"style": VERBOSE_STYLE, "max_tokens": 600},
    "verbose_cap_60":  {"style": VERBOSE_STYLE, "max_tokens": 60},
    "concise":         {"style": CONCISE_STYLE, "max_tokens": 600},
    "concise_cap_120": {"style": CONCISE_STYLE, "max_tokens": 120},
}
LENGTH_QS = ["How does the SLA clock work when Larkspur is waiting on the client?",
             "What does the Professional tier include?",
             "How should a client report a P1 issue, and why?"]

length_records = []
if CONCISE_STYLE.strip():
    for q in LENGTH_QS:
        for name, cfg in CONDITIONS.items():
            rec = await measure_stream(MOE, prefix_first(WARM_TAG, f"{q}\n\n{cfg['style']}"), max_tokens=cfg["max_tokens"])
            rec.update(condition=name, question=q)
            length_records.append(rec)
            await asyncio.sleep(LONG_PAUSE_S)
else:
    print("Write CONCISE_STYLE first.")
df_len = pd.DataFrame(length_records)

In [ ]:
len_table = pd.DataFrame()
if len(df_len) and df_len["ok"].any():
    df_len["cost_usd"] = [request_cost(r.model, r.prompt_tokens, r.completion_tokens) for r in df_len.itertuples()]
    ok = df_len[df_len.ok]
    len_table = ok.groupby("condition").agg(
        out_tokens_p50=("completion_tokens", "median"), total_s_p50=("total_s", "median"), ttft_s_p50=("ttft_s", "median"),
        cost_per_1k_requests_usd=("cost_usd", lambda s: 1000 * s.mean()),
        truncated_share=("finish_reason", lambda s: (s == "length").mean())).reindex(list(CONDITIONS)).reset_index()
    display(len_table.round(4))
    print("How each answer ENDS (look for cut-off sentences):")
    for r in ok[ok.question == LENGTH_QS[1]].itertuples():
        print(f"  {r.condition:16} ...{r.text.strip()[-90:]!r}")
    corr = ok["completion_tokens"].corr(ok["total_s"]) if len(ok) > 2 else None
    print("correlation(output tokens, total latency):", None if corr is None else round(corr, 3))

    cond_colors = dict(zip(CONDITIONS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]))
    fig, ax = plt.subplots(figsize=(7, 3.6))
    for name in CONDITIONS:
        g = ok[ok.condition == name]
        ax.scatter(g.completion_tokens, g.total_s, color=cond_colors[name], s=45, label=name)
    ax.set_xlabel("output tokens"); ax.set_ylabel("total latency (s)"); ax.set_title("Latency follows output length")
    ax.grid(alpha=0.3); ax.legend(fontsize=8); ax.set_xlim(left=0); ax.set_ylim(bottom=0); plt.show()

**Reading the output:** one table row per condition. `out_tokens_p50` is the median answer length in tokens, `total_s_p50` and `ttft_s_p50` are median timings, `cost_per_1k_requests_usd` is what 1,000 such requests would cost at the hypothetical prices, and `truncated_share` is the share of answers cut off by `max_tokens` (1.0 means all of them). The printed endings let you see what a cut-off answer looks like. The **correlation** is a number from -1 to 1; close to 1 means longer answers almost always took longer. In the plot, each dot is one answer: its length across, its total latency up, colored by condition.

In [ ]:
def lv(cond, col):
    g = len_table[len_table.condition == cond] if len(len_table) else []
    return float(g[col].iloc[0]) if len(g) and pd.notna(g[col].iloc[0]) else None

ck.check("length_conditions_ran", len(df_len) and df_len["ok"].sum() >= 12, 2,
         "All 4 conditions x 3 questions should succeed. Write CONCISE_STYLE and re-run.")
v, c = lv("verbose", "out_tokens_p50"), lv("concise", "out_tokens_p50")
ck.check("concise_is_shorter", bool(v and c and c < 0.7 * v), 2,
         "The concise instruction should cut median output tokens well below 'verbose'. Make CONCISE_STYLE stricter.")
for cond in CONDITIONS:
    for col, short in [("out_tokens_p50", "tok"), ("total_s_p50", "lat")]:
        if lv(cond, col) is not None:
            ck.record(f"len_{short}_{cond}", lv(cond, col))

**TODO (decide):** you are choosing the answer-length setting for Larkspur's technicians. Using the table (tokens, latency, cost per 1,000 requests, truncated share) and the printed endings, set `SHIP_CONDITION` to the condition you would ship (one of `"verbose"`, `"verbose_cap_60"`, `"concise"`, `"concise_cap_120"`) and `SAFETY_CAP` to the `max_tokens` value you would set in production as a safety limit. Look at the cost column too: with about 6,000 prompt tokens per request, how much of the cost does a shorter answer actually save? Explain your choice in Q2.

In [ ]:
# TODO: your decision
SHIP_CONDITION = None     # one of the four condition names, as a string
SAFETY_CAP = None         # a whole number of tokens

In [ ]:
ck.check("ship_decision", SHIP_CONDITION in CONDITIONS and isinstance(SAFETY_CAP, int) and 0 < SAFETY_CAP <= 4000, 1,
         "Set SHIP_CONDITION to one of the four condition names and SAFETY_CAP to a whole number of tokens (for example 200).")
if SHIP_CONDITION in CONDITIONS:
    ck.record("ship_condition", SHIP_CONDITION)
if isinstance(SAFETY_CAP, int):
    ck.record("safety_cap", SAFETY_CAP)

**What about routing?** Another way to save money is to send easy questions ("what is the PagerLine number?") to the small, cheap `qwen-small` and only hard ones to `qwen-moe`; look at the gap between their prices in `PRICE_PER_M` above. That is **model routing**, which the book treats as part of the application architecture in Chapter 10, so you build and evaluate a real router in Lab 15. This week stays with the Chapter 9 levers.

### A4. Quantization: a sizing decision (no GPU work)

Chapter 9's "Model compression" section calls weight-only quantization "by far the most popular approach" (the details are in Chapter 7): store weights with fewer bits. In Lab 2 you did the memory math; here you connect it to **speed** and use it to make a hardware decision. Because decode is memory-bandwidth-bound (Lab 12), halving the bytes per weight roughly halves the bytes read per token, which raises the decode ceiling, while quality can drop, especially at 4 bits and below.

The cluster already made these choices for you:

| Alias | Model | Precision on the cluster | Weights in memory | Runs on |
|---|---|---|---|---|
| `qwen-moe` | Qwen3.6-35B-A3B (MoE, 35B total / ~3B active) | 8-bit | ~36 GB | Mac #1 and Mac #2 (one copy each, load-balanced) |
| `qwen-dense` | Qwen3.8-27B (dense) | 4-bit | ~16 GB | Mac #3 |
| `gemma-moe` | Gemma 4 26B-A4B (MoE, ~25B total / ~4B active) | 4-bit | ~16.5 GB | Mac #3 |
| `qwen-small` | Qwen3.5-4B (dense) | 8-bit | ~5 GB | Mac #3 |
| `embed` | Qwen3-Embedding-0.6B | 16-bit | ~1.2 GB | Mac #3 |
| `rerank` | Qwen3-Reranker-0.6B | 4-bit | ~0.3 GB | Mac #3 |

Each Mac Studio M3 Ultra: 96 GB unified memory, about 72 GB usable by the GPU by default, about 819 GB/s memory bandwidth, up to 8 requests at once per Mac. Mac #3 holds about 40 GB of weights, the rest is room for KV cache. Weights sizes are approximate (the "~36 GB" includes the extra multi-token-prediction layers). Real quantized files are a bit larger than parameters x bits / 8, because some parts (such as the embeddings and the quantization scales) are kept at higher precision, so `qwen-dense` at 4 bits is ~16 GB here while `weight_gb(27, 4)` gives 13.5.

Notice the layout: Macs #1 and #2 each serve only `qwen-moe`, while Mac #3 holds five models at once, and every Mac also needs free memory for the KV caches of up to 8 requests at a time. So the real question is not only "does one model fit in 72 GB?" but "do all the models on that Mac fit, with room to spare, and how fast can each one decode?"

`weight_gb()` is provided: you wrote it in Lab 02. The next cell shows weight memory and the bandwidth-bound decode ceiling (819 GB/s, active parameters only) at 16, 8 and 4 bits. Just run it and read it.

In [ ]:
def weight_gb(params_billion, bits):
    """Memory for the weights alone, in GB: parameters x bits / 8 bits per byte. (You wrote this in Lab 02.)"""
    return params_billion * bits / 8

PEAK_BW_GB_S = 819
SPECS = {MOE: (35, 3), DENSE: (27, 27), SMALL: (4, 4)}      # (total params B, active params B)
quant_rows = []
for m, (total, active) in SPECS.items():
    for bits in (16, 8, 4):
        w, a = weight_gb(total, bits), weight_gb(active, bits)
        quant_rows.append({"model": m, "bits": bits, "weights_gb": w,
                           "fits_72gb": w < 72, "share_of_72gb_pct": 100 * w / 72,
                           "decode_ceiling_tok_s": PEAK_BW_GB_S / a})
df_quant = pd.DataFrame(quant_rows)
display(df_quant.round(1))
moe_decode = None
if len(df_len) and df_len["ok"].any():
    moe_decode = 1 / df_len[df_len.ok]["tpot_s"].dropna().median()
    print(f"Your measured qwen-moe decode speed this week (A3): {moe_decode:.0f} tokens/s (served at ~8 bits)")
if moe_decode:
    ck.record("moe_decode_tok_s", float(moe_decode))

**Reading the table:** one row per model and precision (`bits`). `weights_gb` is the memory for the weights alone. `fits_72gb` says whether that one model alone fits in the GPU share of one Mac, and `share_of_72gb_pct` how much of that share it would take; neither counts the KV cache or the other models on the same Mac (see the cluster table). `decode_ceiling_tok_s` is the fastest decode that 819 GB/s of bandwidth allows at that precision, counting active parameters only.

Your measured decode speed (printed under the table) will be well below the ceiling: the ceiling ignores KV-cache reads, software overhead and other users. Lab 12's MBU measured that gap. Compare **ratios** between precisions (for example a model's 8-bit vs 4-bit ceiling), not absolute numbers.

**The sizing decision.** Larkspur wants to run its help-desk assistant on its own hardware and has budget for **one** Mac Studio like the course's (about 72 GB usable by the GPU). It must hold, at the same time:

- `qwen-moe` (the main assistant model),
- `qwen-dense` (for the harder SLA and pricing questions),
- `qwen-small` (cheap model for simple questions),
- `embed` and `rerank` for retrieval (together about 1.5 GB; leave them as they are),
- and at least **17 GB free for KV cache**: about 8 requests at once with long (32K-token) prompts on the dense model, using Lab 02's KV table (about 2.15 GB each).

You choose the precision (16, 8 or 4 bits) for each of the three chat models. Larkspur's rule: **use as many bits as you can** (fewer bits means more quality risk), but the plan must fit with the headroom. There is more than one plan that fits, so you will have to decide which model to squeeze, using the decode ceilings above.

**TODO:** fill in `PLAN` with one of 16, 8 or 4 for each model, run the cell, and adjust until it fits. Then write one or two sentences in `PLAN_REASON`: why you squeezed the model you did (think about which one gains the most speed from fewer bits, and which one you can least afford to lose quality on).

In [ ]:
GPU_LIMIT_GB = 72
KV_HEADROOM_GB = 17
RETRIEVAL_GB = 1.5          # embed + rerank, unchanged

# TODO: choose 16, 8 or 4 bits for each chat model (the starting values do not fit).
PLAN = {MOE: 16, DENSE: 16, SMALL: 16}
# TODO: one or two sentences: which model did you squeeze, and why that one?
PLAN_REASON = ""

In [ ]:
plan_rows = []
for m, bits in PLAN.items():
    total, active = SPECS[m]
    plan_rows.append({"model": m, "bits": bits, "weights_gb": weight_gb(total, bits),
                      "decode_ceiling_tok_s": PEAK_BW_GB_S / weight_gb(active, bits)})
df_plan = pd.DataFrame(plan_rows)
plan_weights = df_plan["weights_gb"].sum() + RETRIEVAL_GB
plan_free = GPU_LIMIT_GB - plan_weights
display(df_plan.round(1))
print(f"Weights on the Mac: {plan_weights:.1f} GB   free for KV cache: {plan_free:.1f} GB   (need at least {KV_HEADROOM_GB} GB)")
plan_valid = set(PLAN) == {MOE, DENSE, SMALL} and all(b in (4, 8, 16) for b in PLAN.values())
plan_fits = plan_valid and plan_free >= KV_HEADROOM_GB
if plan_valid and not plan_fits:
    print("Does not fit yet: lower the bits of one or more models.")
# Could any single model get more bits and still fit? (Larkspur's "as many bits as you can" rule; recorded, not graded)
upgrades = [m for m, b in PLAN.items() if plan_fits and b < 16
            and plan_free - (weight_gb(SPECS[m][0], 2 * b) - weight_gb(SPECS[m][0], b)) >= KV_HEADROOM_GB]
if upgrades:
    print("This fits, but these models could still get more bits:", upgrades)
ck.check("sizing_plan_fits", plan_fits, 2,
         "Each PLAN value must be 16, 8 or 4, and weights + 1.5 GB must leave at least 17 GB free under 72 GB. "
         "Use the weights_gb column of the table above to see which models are the big ones.")
ck.check("sizing_reason", len(PLAN_REASON.strip()) >= 30, 1,
         "Write one or two sentences in PLAN_REASON explaining which model you squeezed and why.")
if plan_valid:
    for m, b in PLAN.items():
        ck.record(f"plan_bits_{m.replace('qwen-', '')}", int(b))
    ck.record("plan_free_gb", round(float(plan_free), 1))
    ck.record("plan_could_upgrade", upgrades)

# Optional: paste your Lab 2 numbers here so the analysis can compare them.
LAB2_NOTES = ""

**Reading the output:** one row per chat model in your plan, with its weight memory and decode ceiling at the precision you chose. The printed line adds the retrieval models and shows how much of the 72 GB is left for the KV cache. Compare your plan with what the course cluster actually does (the table at the top of A4): it uses three Macs, so it can afford more bits than a single box.

## Part B - Stretch (optional, bonus): a response cache in front of the model

Prompt caching saves prefill. A **response cache** skips the model entirely when the same question comes back. Chapter 10 discusses exact and semantic caches as part of AI application architecture ("Step 4. Reduce Latency with Caches": "Exact caching" and "Semantic caching"); we try one here because it is the biggest latency win of all, if it is right.

- **Exact cache:** normalize the question (lowercase, strip punctuation) and look it up in a dict. Never wrong, rarely hits.
- **Semantic cache:** embed the question with `embed` (turn it into a list of numbers that captures its meaning, as in Lab 08) and reuse a stored answer if the cosine similarity to a past question is at or above a threshold. Cosine similarity scores how alike two embeddings are: 1.0 means they point the same way, near 0 means unrelated. Hits paraphrases, but can return the **wrong** answer for a question that only looks similar ("Premier P1 response time" vs "Professional P1 response time"). That is a **false hit**.

`helpdesk_replay.jsonl` holds 50 help-desk questions in arrival order: originals, exact repeats, paraphrases, and near-misses. Each has an `intent` label, so we can tell a correct hit from a false one.

Set `RUN_PART_B = True` to run it (about 25-40 requests, one per cache miss). **TODO A and B** are in `lookup()`; the replay runs only after both unit tests pass.

In [ ]:
RUN_PART_B = False      # set True to do the optional stretch

In [ ]:
def normalize(q):
    return " ".join(re.sub(r"[^a-z0-9 ]", " ", q.lower()).split())


class ResponseCache:
    def __init__(self, threshold=0.85):
        self.threshold = threshold
        self.exact = {}         # normalized question -> entry
        self.vectors = []       # unit-length embedding of each stored question
        self.entries = []       # entry dicts, same order as vectors

    def add(self, question, vector, answer, intent=None):
        entry = {"question": question, "answer": answer, "intent": intent}
        self.exact[normalize(question)] = entry
        v = np.asarray(vector, dtype=float)
        self.vectors.append(v / np.linalg.norm(v))
        self.entries.append(entry)

    def lookup(self, question, vector):
        """Return (entry, kind, score). kind is 'exact', 'semantic' or 'miss'."""
        # TODO A (exact): look the question up in self.exact by its normalize()d form; on a match,
        #        return that entry as an "exact" hit with score 1.0.

        if not self.vectors:
            return None, "miss", 0.0
        # TODO B (semantic): scale `vector` to unit length (as add() does), then get its cosine similarity with
        #        every stored vector at once (np.vstack(self.vectors) and a dot product). Take the best match
        #        (np.argmax). If its score reaches self.threshold, return its entry as a "semantic" hit with that
        #        score; otherwise return a miss (no entry) with that score.
        return None, "miss", 0.0

In [ ]:
# Unit tests on toy vectors (no requests).
_c = ResponseCache(threshold=0.9)
_c.add("What are your hours?", [1, 0, 0], "7-6", "hours")
_c.add("What is the PagerLine number?", [0, 1, 0], "555-0142", "pager")
e1, k1, _ = _c.lookup("what are your HOURS", [0, 0, 1])
e2, k2, s2 = _c.lookup("When are you open?", [0.95, 0.1, 0])
e3, k3, _ = _c.lookup("Something else", [0.5, 0.5, 0.7])
ckb.check("exact_cache", k1 == "exact" and e1 and e1["intent"] == "hours", 2,
          "TODO A: a question that normalizes to a stored one must return kind 'exact'.")
ckb.check("semantic_lookup", k2 == "semantic" and e2 and e2["intent"] == "hours" and k3 == "miss", 3,
          "TODO B: cosine >= threshold should be a 'semantic' hit; a dissimilar vector must be a 'miss'.")
LOOKUP_OK = bool(k1 == "exact" and e1 and e1["intent"] == "hours" and k2 == "semantic" and e2 and e2["intent"] == "hours"
                 and k3 == "miss")

In [ ]:
def find_replay():
    for p in [Path("helpdesk_replay.jsonl"), corpus_dir().parents[1] / "labs" / "lab13-serving-optimization" / "helpdesk_replay.jsonl"]:
        if p.exists():
            return p
    raise FileNotFoundError("helpdesk_replay.jsonl not found; open this notebook from its lab folder.")

REPLAY = [json.loads(line) for line in find_replay().read_text().splitlines() if line.strip()]
print(len(REPLAY), "questions;", pd.Series([r["kind"] for r in REPLAY]).value_counts().to_dict())
THRESHOLD = 0.85
replay_rows, VECS = [], None

if RUN_PART_B and not LOOKUP_OK:
    print("Finish TODO A and B first (both unit tests above must pass). Skipping the replay so it doesn't send 52 requests for nothing.")
elif RUN_PART_B:
    # One batch call embeds all 50 questions (in production each question is embedded when it arrives).
    VECS = [d.embedding for d in client.embeddings.create(model=EMBED, input=[r["question"] for r in REPLAY]).data]
    BUDGET["sent"] += 1
    t0 = time.perf_counter()
    client.embeddings.create(model=EMBED, input=REPLAY[0]["question"]); BUDGET["sent"] += 1
    embed_latency = time.perf_counter() - t0            # what a cache hit costs: one embedding call
    cache, seen = ResponseCache(THRESHOLD), set()
    for item, vec in zip(REPLAY, VECS):
        entry, kind, score = cache.lookup(item["question"], vec)
        row = {**item, "kind_hit": kind, "score": score, "should_hit": item["intent"] in seen}
        if entry is not None:
            row.update(hit=True, false_hit=entry["intent"] != item["intent"], latency_s=embed_latency,
                       matched=entry["question"])
        else:
            rec = await measure_stream(MOE, prefix_first(WARM_TAG, item["question"]), max_tokens=120)
            cache.add(item["question"], vec, rec["text"], item["intent"])
            row.update(hit=False, false_hit=False, latency_s=(rec["total_s"] or 0) + embed_latency, matched=None)
            await asyncio.sleep(LONG_PAUSE_S)
        seen.add(item["intent"])
        replay_rows.append(row)
df_replay = pd.DataFrame(replay_rows)
if len(df_replay):
    hits = df_replay.hit.sum()
    print(f"hit rate {hits / len(df_replay):.0%} ({hits}/50), false hits {df_replay.false_hit.sum()}, "
          f"missed chances {(df_replay.should_hit & ~df_replay.hit).sum()}, best possible hit rate {df_replay.should_hit.mean():.0%}")
    print(f"mean latency with cache {df_replay.latency_s.mean():.2f} s vs misses only {df_replay[~df_replay.hit].latency_s.mean():.2f} s")
    display(df_replay[df_replay.hit][["id", "question", "kind_hit", "score", "false_hit", "matched"]].round(3))

**Reading the output:** the first printed line counts cache hits, false hits (the cache returned the answer to a different question), missed chances (an answer to the same question was stored, but the cache missed it), and the best hit rate possible on this replay. The second line compares average latency with the cache against the requests that had to go to the model. The table lists every hit: `kind_hit` is `exact` or `semantic`, `score` is the cosine similarity, `false_hit` marks wrong hits, and `matched` is the stored question it was matched to.

In [ ]:
# Threshold sweep, offline: replays the same 50 questions with the embeddings you already have (no requests).
sweep = pd.DataFrame()
if RUN_PART_B and VECS is not None:
    rows = []
    for th in np.round(np.arange(0.60, 0.99, 0.02), 2):
        c, n_hit, n_false = ResponseCache(th), 0, 0
        for item, vec in zip(REPLAY, VECS):
            entry, kind, _ = c.lookup(item["question"], vec)
            if entry is None:
                c.add(item["question"], vec, "", item["intent"])
            else:
                n_hit += 1; n_false += entry["intent"] != item["intent"]
        rows.append({"threshold": th, "hit_rate": n_hit / len(REPLAY), "false_hit_rate": n_false / len(REPLAY)})
    sweep = pd.DataFrame(rows)
    fig, ax = plt.subplots(figsize=(7, 3.6))
    ax.plot(sweep.threshold, sweep.hit_rate, "-o", color="#2a78d6", lw=2, ms=5, label="hit rate (all hits)")
    ax.plot(sweep.threshold, sweep.false_hit_rate, "-o", color="#eb6834", lw=2, ms=5, label="false-hit rate")
    ax.axvline(THRESHOLD, color="#888888", ls=":", lw=1)
    ax.set_xlabel("cosine threshold"); ax.set_ylabel("share of the 50 questions"); ax.set_ylim(0, 1)
    ax.set_title("Semantic cache: more hits means more wrong hits"); ax.grid(alpha=0.3); ax.legend(); plt.show()

**Reading the plot:** across is the similarity threshold; up is the share of the 50 questions. The blue line is all hits (right and wrong), the orange line only the false hits, and the dotted line marks the threshold you used. A good threshold keeps the blue line high while the orange line stays near zero.

In [ ]:
if RUN_PART_B:
    replay_ok = len(df_replay) == 50 and bool(df_replay.hit.any())
    ckb.check("replay_complete", replay_ok, 2,
              "The replay should process all 50 questions with a working cache (at least one hit). Finish TODO A and B.")
    ckb.check("threshold_sweep", len(sweep) >= 10 and float(sweep.hit_rate.max()) > 0, 1,
              "Run the sweep cell after TODO A and B work; at least one threshold should give hits.")
    ckb.check("false_hits_measured", replay_ok and "false_hit" in df_replay and "false_hit_rate" in sweep, 2,
              "The replay and the sweep must both track false hits (needs a working cache).")
    if len(df_replay):
        ck.record("cache_hit_rate", float(df_replay.hit.mean()))
        ck.record("cache_false_hits", int(df_replay.false_hit.sum()))
        ck.record("cache_threshold", THRESHOLD)
bonus_got, bonus_max = ckb.score()
ck.record("bonus_score", bonus_got)
ck.record("bonus_max", 10)
ck.record("requests_sent", BUDGET["sent"])
print(f"Bonus (Part B) auto-check points: {bonus_got}/10   |   requests this notebook sent: {BUDGET['sent']}")

## Written analysis

Use **your own numbers** and cite Chapter 9 by section name or concept where asked.

**Q1. Prompt caching and prefix order.** Report median TTFT for cold, warm and question-first ("last") requests, and your speedup. Explain *why* prefix order matters, using what the KV cache stores (Chapter 9, "Prompt caching" and the KV cache discussion under attention optimization). What did the `backend` column show for warm `qwen-moe` requests, and how does `qwen-small` (one Mac) compare? What would you change in the gateway or in Larkspur's prompt layout to get more cache hits? (For the gateway: how could it make requests with the same prefix land on the same Mac?) (Think about what happens if someone puts today's date or the user's name at the top of the system prompt.)

*Your answer:*

**Q2. Batching and output length.** Using your A2 and A3 tables: how much wall time did concurrency save for 8 tickets, and what happened to per-request latency? Relate this to continuous batching (Chapter 9, "Batching"). Then compare `max_tokens` with the concise instruction on output tokens, latency, cost per 1,000 requests, and answer quality (quote one truncated ending). Which would you ship for Larkspur's technicians, and with what safety cap?

*Your answer:*

**Q3. Quantization and your sizing plan.** Report your `PLAN` (bits per model), the GB left for KV cache, and the decode ceiling of each model in your plan. Which model did you squeeze, and why that one rather than the other? Going from 8 to 4 bits doubles the decode ceiling of every model; use the actual tokens per second in your table to explain why that doubling matters much more for one model than the other, and compare the ceiling with the `qwen-moe` decode speed you measured in A3. Then explain why the course cluster can serve `qwen-moe` at 8 bits on its own Macs while it puts `qwen-dense` at 4 bits on the shared Mac #3. Cite Chapter 9's "Model compression" discussion of quantization, and use your Lab 2 numbers if you have them.

*Your answer:*

**Q4. Recommendation for Larkspur.** Rank the optimizations you tested (prompt caching, concurrency/batching, output length control, quantization; and the response cache if you did Part B) by how much they would help Larkspur's help-desk assistant. Back each rank with one of your numbers (for quantization you measured nothing this week, so an estimate from your A4 tables is fine), and say what each one costs (complexity, quality risk, or a constraint it puts on prompt design). If you did Part B, include your hit rate and false-hit count and the threshold you would choose.

*Your answer:*

## AI-use reflection

Answer briefly but specifically.

1. What did you use AI assistance for in this lab (tools, which steps)?

*Your answer:*

2. Where did the AI help most, and where was it wrong or unhelpful? How did you notice?

*Your answer:*

3. What is one thing you now understand that you could explain without AI help?

*Your answer:*

4. Lab-specific: did an AI assistant suggest a prompt layout or code change that would quietly break prompt caching (for example a timestamp, request id, or the question at the top of the prompt)? How would you spot that from your measurements?

*Your answer:*

## Submit

Run this last, save, upload the .ipynb.

In [ ]:
ck.report();